# Build the benchmark sets: `papers_benchset_v1.parquet` -> three named datasets

Sibling of `04_feature_engineering.ipynb`, same job — assemble every already-validated,
fold-independent feature block into one table, and model, score, tune and split nothing.
The governing rule is unchanged: a column belongs here only if its value would be the
same regardless of which rows land in a training fold. PCA, scalers, TF-IDF vocabularies
and imputers stay out.

What this notebook adds that the 6-silo version didn't need, because 28 silos at
0.16%-78% prevalence force the question: **which collections can carry a fitted model at
all, and how do you hold some of them back without leaking?**

| output | collections | for |
|---|---|---|
| `benchset_v1_small_test.parquet` | those held out of the fittable tier | zero-shot / ranking evaluation only |
| `benchset_v1_large_set_a.parquet` | development half | iterate freely |
| `benchset_v1_large_set_b.parquet` | held-out half | touch once |

**A/B is dev vs held-out, both used within-silo.** It is not a cross-silo train/test
split: `CONTEXT.md` §1 forbids a pooled model shipping, and `03_eda_full_benchset_v1`
§14 declined to run LOGO for that reason. The split exists to stop the *method* being
overfitted to the collections it was developed on, which is why what gets minimised is
leakage across the A/B boundary — shared papers first, topical similarity second.

Three inclusion rules, applied in this order, each measured rather than assumed:

1. **Drop papers with no abstract.** Costs 2.6% of rows and 3.1% of positives.
2. **Tier at 40 positives**, then *prove* the threshold by running the folds.
3. **Quarantine what isn't a screening task**, regardless of how many positives it has.

## 1. Config and schema check

Same reasoning as `04_feature_engineering.ipynb` §1: `fold_pipeline_utils.validate_schema`
reads its required-column list out of a fold-pipeline-shaped `CONFIG`, so reusing it here
would mean building a fake CONFIG to satisfy its shape. Explicit check instead — same
fail-loudly-here goal, right-sized tool.

In [1]:
import itertools
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.model_selection import StratifiedGroupKFold

sys.path.insert(0, "../../scripts")
from embed_benchsets import load_brief_vectors, load_paper_vectors
from fold_pipeline_utils import derive_first_author
from lexical_features import build_lexical_features, derangements

DATA_PATH = Path("../../data/processed/papers_benchset_v1.parquet")
OUT_DIR = Path("../../data/processed")
REPORT_DIR = Path("../../reports")

MODEL_KEYS = ["jasper", "qwen4b"]

# Asserted at write time, exactly as 04_feature_engineering.ipynb §10 asserts its own
# expected_widths - a silently narrow embedding block is the failure this catches.
EXPECTED_DIMS = {"jasper": 2048, "qwen4b": 2560}

# Both brief variants are carried. §8.4 of 03_eda_full_benchset_v1.ipynb measured the gap
# between them at ~0.005 ROC-AUC against a ~0.03 noise floor, so `full` is not obviously
# cheating - but `pre_screening` is the one a customer could actually write on day one,
# and having both here means downstream work can settle that without re-embedding.
BRIEF_VARIANT_PREFIX = {"full": "cos_brief", "pre_screening": "cos_briefpre"}

MIN_POSITIVES = 40   # 5 folds x 8 positives; verified against real folds in §5, not assumed
N_FOLDS = 5
MIN_POS_PER_TEST_FOLD = 5

# Excluded from the fittable tier regardless of positive count. Not a data-quality
# judgement - a scope one, and it travels with its reason so the choice stays auditable.
QUARANTINE = {
    "roadfreight_metareview": (
        "review-of-reviews at 78% prevalence over 132 rows: the screening decision is "
        "'is this a review?', not 'is this relevant?'. Already out of every headline "
        "average in 02/03_eda_*_benchset_v1."
    ),
}

BUILD_CONTROL = False  # True re-runs Tier 1b against deliberately wrong briefs (lexctl_*)

# Balance constraints for the A/B search. Positives are the scarce resource so they get
# the tighter bound; rows are allowed to drift further because synergy_walker_2018 is
# 32% of the fittable corpus by itself and no partition can be tight on both.
MAX_COUNT_DIFF = 1
MAX_POS_IMBALANCE = 0.12
MAX_ROW_IMBALANCE = 0.15

REQUIRED_COLS = [
    "row_key", "paper_id", "use_case_key", "triage_label", "label_positive",
    "authors", "title", "abstract", "doi", "year", "citation_count",
    "has_abstract", "exported_at",
    "objective", "problem_statement", "terms_must_include", "terms_nice_to_have",
    "terms_exclude", "domain_industry", "domain_application", "domain_technology_focus",
]

In [2]:
df_raw = pd.read_parquet(DATA_PATH)

missing = [c for c in REQUIRED_COLS if c not in df_raw.columns]
if missing:
    raise ValueError(
        f"{DATA_PATH.name} is missing column(s) {missing} this notebook depends on - "
        f"re-run 01_data_compile_benchset_v1.ipynb or update REQUIRED_COLS. Columns "
        f"present: {sorted(df_raw.columns)}"
    )
print(f"Loaded {DATA_PATH.name}: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} cols, schema OK.")
print(f"{df_raw['use_case_key'].nunique()} collections, "
      f"{int(df_raw['label_positive'].sum()):,} positives "
      f"({df_raw['label_positive'].mean():.2%} pooled prevalence).")

Loaded papers_benchset_v1.parquet: 181,199 rows x 24 cols, schema OK.
28 collections, 3,374 positives (1.86% pooled prevalence).


## 2. Filter: labels, then the abstract drop

`04_feature_engineering.ipynb` §2 spends its length on `pass` and never-triaged rows.
Neither exists here — this corpus is 100% `positive`/`negative`, which
`01_data_compile_benchset_v1.ipynb` already asserted against the manifest. So that
handling is absent by design, and it's asserted rather than assumed.

The real filter is **papers with no abstract**. A title-only row still produces an
embedding and a BM25 score, but from ~12 words instead of ~200, so every text feature
means something different for it. Dropping them makes one inclusion rule that can be
stated in a sentence, at a measured cost of 2.6% of rows.

The cost is reported per collection rather than pooled, because a pooled 2.6% would hide
the one collection where this drop is **not label-neutral** — see the next cell.

In [3]:
n_start = len(df_raw)
pos_start = int(df_raw["label_positive"].sum())

label_counts = df_raw["triage_label"].value_counts(dropna=False)
print("triage_label counts:")
print(label_counts.to_string())
assert df_raw["triage_label"].notna().all(), "null triage_label - 01 should have caught this"
assert set(df_raw["triage_label"].unique()) == {"positive", "negative"}, (
    f"unexpected label vocabulary {set(df_raw['triage_label'].unique())} - this corpus is "
    f"documented as binary-complete, so anything else means the input changed"
)
print("\nNo 'pass' or never-triaged rows: 04's label-filtering step is absent by design here.")

df = df_raw[df_raw["has_abstract"]].reset_index(drop=True)
n_after_abstract = len(df)
pos_after_abstract = int(df["label_positive"].sum())

print(f"\nDropped {n_start - n_after_abstract:,} rows with no abstract "
      f"({(n_start - n_after_abstract) / n_start:.1%}), of which "
      f"{pos_start - pos_after_abstract} were positive "
      f"({(pos_start - pos_after_abstract) / pos_start:.1%} of all positives).")
print(f"Pooled prevalence {pos_start / n_start:.2%} -> {pos_after_abstract / n_after_abstract:.2%}.")
print(f"Rows: {n_start:,} -> {n_after_abstract:,}   positives: {pos_start:,} -> {pos_after_abstract:,}")

triage_label counts:
triage_label
negative    177825
positive      3374

No 'pass' or never-triaged rows: 04's label-filtering step is absent by design here.

Dropped 4,733 rows with no abstract (2.6%), of which 104 were positive (3.1% of all positives).
Pooled prevalence 1.86% -> 1.85%.
Rows: 181,199 -> 176,466   positives: 3,374 -> 3,270


### 2.1 Where this drop is *not* label-neutral — a hard finding, and a caveat that has to travel

Dropping abstract-less papers is only harmless if missingness is unrelated to the label.
It mostly is: in 17 of the 25 affected collections the dropped rows contain **zero**
positives, and the pooled prevalence barely moves.

`synergy_walker_2018` is the exception, and it is the corpus's largest collection.

In [4]:
rows = []
for uc, sub in df_raw.groupby("use_case_key"):
    dropped = sub[~sub["has_abstract"]]
    if not len(dropped):
        continue
    kept = sub[sub["has_abstract"]]
    prev_dropped = float(dropped["label_positive"].mean())
    prev_kept = float(kept["label_positive"].mean())
    rows.append({
        "use_case_key": uc,
        "n_dropped": len(dropped),
        "pos_dropped": int(dropped["label_positive"].sum()),
        "prev_dropped": prev_dropped,
        "prev_kept": prev_kept,
        "enrichment": prev_dropped / prev_kept if prev_kept else np.nan,
    })
neutrality = pd.DataFrame(rows).sort_values("enrichment", ascending=False)
print("Prevalence among the DROPPED rows vs the KEPT rows, per collection:")
print(neutrality.round(4).to_string(index=False))

n_clean = int((neutrality["pos_dropped"] == 0).sum())
print(f"\n{n_clean} of {len(neutrality)} affected collections lose zero positives.")

worst = neutrality.iloc[0]
print(
    f"\nHARD FINDING - {worst['use_case_key']}: its {worst['n_dropped']:,} abstract-less "
    f"rows are {worst['prev_dropped']:.2%} positive against {worst['prev_kept']:.2%} among "
    f"the rows that keep an abstract ({worst['enrichment']:.1f}x enriched). The drop costs "
    f"it {worst['pos_dropped']} positives."
)
print(
    "\nCONSEQUENCE, and it must be quoted wherever this collection's numbers are: the\n"
    "papers removed here are disproportionately the RELEVANT ones, and they are the\n"
    "hardest kind (title only, no abstract to match against). Any score measured on\n"
    "synergy_walker_2018 in these files is therefore OPTIMISTIC relative to the real\n"
    "screening task, which would have had to find those papers too.\n\n"
    "The rule is applied uniformly anyway. A per-collection exception would buy back 81\n"
    "positives at the cost of a dataset whose inclusion criteria can no longer be stated\n"
    "in one sentence - and a caveat you can write down beats an exception you have to\n"
    "remember."
)

Prevalence among the DROPPED rows vs the KEPT rows, per collection:
                   use_case_key  n_dropped  pos_dropped  prev_dropped  prev_kept  enrichment
            synergy_walker_2018       1514           81        0.0535     0.0145      3.6898
           synergy_donners_2021          8            1        0.1250     0.0560      2.2321
         roadfreight_metareview          8            8        1.0000     0.7661      1.3053
         synergy_wassenaar_2017        429            6        0.0140     0.0145      0.9642
      synergy_van_der_valk_2021         10            1        0.1000     0.1231      0.8125
          synergy_meijboom_2021         58            1        0.0172     0.0437      0.3946
          synergy_leenaars_2020        374            5        0.0134     0.0844      0.1584
synergy_appenzeller-herzog_2019        659            1        0.0015     0.0115      0.1324
               synergy_oud_2018         15            0        0.0000     0.0213      0.0000
  

## 3. Dedupe — within `use_case_key` only

Carried unchanged from `04_feature_engineering.ipynb` §3, including the reasoning, which
this corpus makes concrete rather than hypothetical: a paper appearing under two briefs is
two legitimate (brief, paper) pairs, each independently labelled. `03_eda_full_benchset_v1`
§4 measured **130 papers labelled `positive` under one question and `negative` under
another** — deduping globally would destroy exactly that signal.

The corpus already deduped on DOI upstream (its manifest documents it), so the DOI pass
here should find nothing. The title pass is the one that earns its place.

In [5]:
def normalise_title(t):
    return " ".join(str(t).lower().split())


df["_norm_title"] = df["title"].map(normalise_title)
df["_doi"] = df["doi"].fillna("").astype(str).str.strip()
has_doi = df["_doi"] != ""

within_doi = df.duplicated(subset=["use_case_key", "_doi"], keep="first") & has_doi
within_title = df.duplicated(subset=["use_case_key", "_norm_title"], keep="first")
drop_mask = within_doi | within_title

print(f"Within-use_case_key duplicate DOI rows dropped:            {int(within_doi.sum()):,}")
print(f"Within-use_case_key near-duplicate title rows dropped:     {int(within_title.sum()):,}")
print(f"Total dropped: {int(drop_mask.sum()):,} "
      f"({int(df.loc[drop_mask, 'label_positive'].sum())} of them positive)")
print(
    f"\nThe DOI pass finding {int(within_doi.sum())} confirms the corpus's own documented "
    f"DOI-based dedupe. The title pass is what it misses: same paper, two records, "
    f"different or absent DOI."
)

df = df[~drop_mask].drop(columns=["_norm_title", "_doi"]).reset_index(drop=True)
n_after_dedupe = len(df)
pos_after_dedupe = int(df["label_positive"].sum())
print(f"\nRows: {n_after_abstract:,} -> {n_after_dedupe:,}   "
      f"positives: {pos_after_abstract:,} -> {pos_after_dedupe:,}")

Within-use_case_key duplicate DOI rows dropped:            0
Within-use_case_key near-duplicate title rows dropped:     1,147
Total dropped: 1,147 (9 of them positive)

The DOI pass finding 0 confirms the corpus's own documented DOI-based dedupe. The title pass is what it misses: same paper, two records, different or absent DOI.

Rows: 176,466 -> 175,319   positives: 3,270 -> 3,261


## 4. Group key

`derive_first_author` from `scripts/fold_pipeline_utils.py`, reused rather than re-derived
so this notebook's grouping and the fold pipelines' can never drift apart. Rows with no
parseable author each become a unique `no_author_<index>` singleton rather than collapsing
into one giant shared group.

In [6]:
df["first_author"] = derive_first_author(df["authors"])
n_singleton = int(df["first_author"].str.startswith("no_author_").sum())
print(f"first_author derived: {df['first_author'].nunique():,} distinct groups across "
      f"{len(df):,} rows.")
print(f"{n_singleton:,} rows had no parseable author (unique singleton each, never pooled).")
print(f"Largest group: {df['first_author'].value_counts().iloc[0]} rows.")

first_author derived: 132,830 distinct groups across 175,319 rows.
17,288 rows had no parseable author (unique singleton each, never pooled).
Largest group: 68 rows.


## 5. Tier the collections — and prove the threshold instead of trusting it

A collection earns a fitted model only if it can be cross-validated. The working rule is
**≥40 positives**, i.e. 5 folds x 8 — but that arithmetic assumes positives distribute
evenly across folds, and they don't have to: folds are grouped by first author and
stratified on a 1-in-60 label, so a collection can clear 40 and still produce a fold with
almost nothing in it to score.

So the threshold is applied and then **checked against real folds**. A collection that
clears 40 but fails the fold check is a finding, not a silent pass.

In [7]:
by_uc = df.groupby("use_case_key")
tier_table = pd.DataFrame({
    "rows": by_uc.size(),
    "positives": by_uc["label_positive"].sum().astype(int),
    "authors": by_uc["first_author"].nunique(),
})
tier_table["prevalence"] = tier_table["positives"] / tier_table["rows"]

tier_table["tier"] = np.where(
    tier_table.index.isin(QUARANTINE), "small_test",
    np.where(tier_table["positives"] >= MIN_POSITIVES, "large", "small_test"),
)
tier_table["reason"] = np.where(
    tier_table.index.isin(QUARANTINE), "quarantined",
    np.where(tier_table["positives"] >= MIN_POSITIVES, "",
             f"under {MIN_POSITIVES} positives"),
)

print(tier_table.sort_values(["tier", "positives"])
      .round(4).to_string())
print()
for uc, why in QUARANTINE.items():
    if uc in tier_table.index:
        print(f"QUARANTINED {uc} ({tier_table.loc[uc, 'positives']} positives, "
              f"{tier_table.loc[uc, 'prevalence']:.1%} prevalence): {why}")

print()
print(tier_table.groupby("tier").agg(
    collections=("rows", "size"), rows=("rows", "sum"), positives=("positives", "sum")
).to_string())

                                  rows  positives  authors  prevalence        tier              reason
use_case_key                                                                                          
synergy_sep_2021                   270         40      250      0.1481       large                    
synergy_radjenovic_2013           5890         48     4948      0.0081       large                    
synergy_brouwer_2019             37401         62    28177      0.0017       large                    
synergy_van_dis_2020              8938         72     6996      0.0081       large                    
synergy_menon_2022                 969         74      949      0.0764       large                    
synergy_nelson_2002                358         80      319      0.2235       large                    
synergy_van_der_valk_2021          710         87      612      0.1225       large                    
synergy_jeyaraman_2020            1170         96      924      0.0821   

In [8]:
large_keys = sorted(tier_table.index[tier_table["tier"] == "large"])
print(f"Running real {N_FOLDS}-fold StratifiedGroupKFold on each of the "
      f"{len(large_keys)} candidates, grouped by first author:\n")

fold_check = []
for uc in large_keys:
    sub = df[df["use_case_key"] == uc]
    y = sub["label_positive"].to_numpy().astype(int)
    groups = sub["first_author"].to_numpy()
    splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
    per_fold = [int(y[test].sum()) for _, test in splitter.split(sub, y, groups)]
    fold_check.append({"use_case_key": uc, "positives": int(y.sum()),
                       "min_pos_in_a_test_fold": min(per_fold),
                       "folds": per_fold})

fold_frame = pd.DataFrame(fold_check).sort_values("min_pos_in_a_test_fold")
print(fold_frame.to_string(index=False))

failed = fold_frame[fold_frame["min_pos_in_a_test_fold"] < MIN_POS_PER_TEST_FOLD]
if len(failed):
    raise AssertionError(
        f"{len(failed)} collection(s) clear {MIN_POSITIVES} positives but cannot produce "
        f"{N_FOLDS} usable folds: {failed['use_case_key'].tolist()}. Demote them to "
        f"small_test rather than shipping a tier that can't be fitted."
    )
print(
    f"\nAll {len(large_keys)} pass: every test fold carries at least "
    f"{int(fold_frame['min_pos_in_a_test_fold'].min())} positives (floor was "
    f"{MIN_POS_PER_TEST_FOLD}). The {MIN_POSITIVES}-positive threshold is therefore "
    f"demonstrated on this corpus, not inherited."
)

Running real 5-fold StratifiedGroupKFold on each of the 15 candidates, grouped by first author:



             use_case_key  positives  min_pos_in_a_test_fold                     folds
         synergy_sep_2021         40                       8           [8, 8, 8, 8, 8]
  synergy_radjenovic_2013         48                       9         [10, 9, 11, 9, 9]
     synergy_brouwer_2019         62                      12      [12, 12, 13, 13, 12]
       synergy_menon_2022         74                      14      [15, 14, 15, 15, 15]
     synergy_van_dis_2020         72                      14      [14, 15, 15, 14, 14]
      synergy_nelson_2002         80                      15      [16, 16, 17, 16, 15]
synergy_van_der_valk_2021         87                      17      [17, 17, 17, 18, 18]
   synergy_jeyaraman_2020         96                      19      [19, 19, 20, 19, 19]
        synergy_hall_2012        104                      20      [20, 20, 22, 21, 21]
   synergy_wassenaar_2017        105                      21      [21, 21, 21, 21, 21]
       synergy_moran_2021        111       

## 6. Split the fittable tier into A and B

Two ways a held-out set stops being held out, and both are measured rather than assumed:

1. **Shared papers.** 2,092 papers appear in more than one fittable collection. If a paper
   sits in A and in B, decisions made while looking at A were made having seen it.
2. **Topical similarity.** Two collections asking near-identical questions leak judgement
   even with no shared rows — tuning against one is tuning against the other.

16 collections (15 after quarantine) is small enough to solve **exactly**: every one of
the 2^14 partitions is enumerated, filtered to those that are balanced, and ranked by
shared papers crossing the boundary, tie-broken on summed brief-brief cosine. No heuristic,
no seed, no local optimum.

Papers are identified by DOI where present and normalised title otherwise — the same
identity rule §3 deduped with, so "shared" means the same thing in both places.

In [9]:
large_df = df[df["use_case_key"].isin(large_keys)].copy()
large_df["_doi"] = large_df["doi"].fillna("").astype(str).str.strip().str.lower()
large_df["_paper_identity"] = np.where(
    large_df["_doi"] != "",
    "doi:" + large_df["_doi"],
    "ttl:" + large_df["title"].map(normalise_title),
)

spread = large_df.groupby("_paper_identity")["use_case_key"].nunique()
shared_identities = set(spread.index[spread > 1])
print(f"{len(large_keys)} fittable collections, {len(large_df):,} rows, "
      f"{int(large_df['label_positive'].sum()):,} positives.")
print(f"{len(shared_identities):,} distinct papers appear in more than one of them.")

index_of = {uc: i for i, uc in enumerate(large_keys)}
n_uc = len(large_keys)
share_matrix = np.zeros((n_uc, n_uc))
for _, group in large_df[large_df["_paper_identity"].isin(shared_identities)].groupby(
        "_paper_identity"):
    for a, b in itertools.combinations(sorted(group["use_case_key"].unique()), 2):
        share_matrix[index_of[a], index_of[b]] += 1
        share_matrix[index_of[b], index_of[a]] += 1

brief_vectors = load_brief_vectors("qwen4b", "full")
brief_matrix = np.vstack([brief_vectors[uc] for uc in large_keys])
brief_matrix /= np.linalg.norm(brief_matrix, axis=1, keepdims=True)
topic_matrix = brief_matrix @ brief_matrix.T
np.fill_diagonal(topic_matrix, 0.0)

print(f"\nTotal sharing links: {share_matrix.sum() / 2:,.0f}")
print("\nThe pairs any partition has to respect (top 6 by shared papers):")
pairs = sorted(
    ((share_matrix[i, j], topic_matrix[i, j], large_keys[i], large_keys[j])
     for i, j in itertools.combinations(range(n_uc), 2) if share_matrix[i, j]),
    reverse=True,
)
for n_shared, cos, a, b in pairs[:6]:
    print(f"  {n_shared:6.0f} shared | brief cosine {cos:.3f} | {a} + {b}")

15 fittable collections, 144,579 rows, 2,904 positives.
2,092 distinct papers appear in more than one of them.



Total sharing links: 2,112

The pairs any partition has to respect (top 6 by shared papers):
    1100 shared | brief cosine 0.481 | synergy_brouwer_2019 + synergy_van_dis_2020
     634 shared | brief cosine 0.712 | synergy_hall_2012 + synergy_radjenovic_2013
      98 shared | brief cosine 0.265 | synergy_brouwer_2019 + synergy_walker_2018
      63 shared | brief cosine 0.506 | synergy_walker_2018 + synergy_wassenaar_2017
      57 shared | brief cosine 0.360 | synergy_brouwer_2019 + synergy_moran_2021
      30 shared | brief cosine 0.254 | synergy_jeyaraman_2020 + synergy_walker_2018


In [10]:
rows_per = np.array([int((large_df["use_case_key"] == uc).sum()) for uc in large_keys], float)
pos_per = np.array([int(large_df.loc[large_df["use_case_key"] == uc, "label_positive"].sum())
                    for uc in large_keys], float)

candidates = []
for mask in range(1, 2 ** (n_uc - 1)):
    side_a = np.array([(mask >> i) & 1 for i in range(n_uc)], dtype=bool)
    n_a = int(side_a.sum())
    if n_a == 0 or n_a == n_uc:
        continue
    if abs(n_a - (n_uc - n_a)) > MAX_COUNT_DIFF:
        continue
    pos_imbalance = abs(pos_per[side_a].sum() - pos_per[~side_a].sum()) / pos_per.sum()
    row_imbalance = abs(rows_per[side_a].sum() - rows_per[~side_a].sum()) / rows_per.sum()
    if pos_imbalance > MAX_POS_IMBALANCE or row_imbalance > MAX_ROW_IMBALANCE:
        continue
    candidates.append((
        share_matrix[np.ix_(side_a, ~side_a)].sum(),
        topic_matrix[np.ix_(side_a, ~side_a)].sum(),
        pos_imbalance, row_imbalance, mask,
    ))

if not candidates:
    raise AssertionError(
        f"no partition satisfies the balance constraints (count diff <= {MAX_COUNT_DIFF}, "
        f"positives within {MAX_POS_IMBALANCE:.0%}, rows within {MAX_ROW_IMBALANCE:.0%}). "
        f"Loosen them deliberately rather than letting an unbalanced split through."
    )

candidates.sort(key=lambda c: (c[0], c[1]))
cut, topical_cut, pos_imbalance, row_imbalance, best_mask = candidates[0]
set_a_keys = sorted(large_keys[i] for i in range(n_uc) if (best_mask >> i) & 1)
set_b_keys = sorted(k for k in large_keys if k not in set_a_keys)

total_links = share_matrix.sum() / 2
print(f"Enumerated {2 ** (n_uc - 1):,} partitions; {len(candidates):,} met the balance "
      f"constraints.\n")
print(f"BEST: {cut:.0f} of {total_links:,.0f} sharing links cross the A/B boundary "
      f"({1 - cut / total_links:.1%} of all sharing stays inside one side).")
print(f"      summed cross-boundary brief cosine {topical_cut:.2f} | "
      f"positives imbalanced {pos_imbalance:.1%} | rows imbalanced {row_imbalance:.1%}\n")

assignment = {**{k: "large_set_a" for k in set_a_keys},
              **{k: "large_set_b" for k in set_b_keys}}
for name, keys in (("set_a", set_a_keys), ("set_b", set_b_keys)):
    sub = large_df[large_df["use_case_key"].isin(keys)]
    print(f"{name} - {len(keys)} collections, {len(sub):,} rows, "
          f"{int(sub['label_positive'].sum()):,} positives "
          f"({sub['label_positive'].mean():.2%})")
    for uc in sorted(keys, key=lambda k: -pos_per[index_of[k]]):
        print(f"    {uc:34s} {int(rows_per[index_of[uc]]):7,} rows  "
              f"{int(pos_per[index_of[uc]]):4d} pos  "
              f"{pos_per[index_of[uc]] / rows_per[index_of[uc]]:6.2%}")
    print()

Enumerated 16,384 partitions; 792 met the balance constraints.

BEST: 159 of 2,112 sharing links cross the A/B boundary (92.5% of all sharing stays inside one side).
      summed cross-boundary brief cosine 16.28 | positives imbalanced 6.2% | rows imbalanced 13.9%

set_a - 8 collections, 62,229 rows, 1,362 positives (2.19%)
    synergy_leenaars_2020                6,711 rows   576 pos   8.58%
    synergy_muthu_2021                   2,687 rows   334 pos  12.43%
    synergy_moran_2021                   5,154 rows   111 pos   2.15%
    synergy_van_der_valk_2021              710 rows    87 pos  12.25%
    synergy_nelson_2002                    358 rows    80 pos  22.35%
    synergy_van_dis_2020                 8,938 rows    72 pos   0.81%
    synergy_brouwer_2019                37,401 rows    62 pos   0.17%
    synergy_sep_2021                       270 rows    40 pos  14.81%



set_b - 7 collections, 82,350 rows, 1,542 positives (1.87%)
    synergy_walker_2018                 46,519 rows   675 pos   1.45%
    nykvist_evcharging                  11,853 rows   440 pos   3.71%
    synergy_wassenaar_2017               7,225 rows   105 pos   1.45%
    synergy_hall_2012                    8,724 rows   104 pos   1.19%
    synergy_jeyaraman_2020               1,170 rows    96 pos   8.21%
    synergy_menon_2022                     969 rows    74 pos   7.64%
    synergy_radjenovic_2013              5,890 rows    48 pos   0.81%



### 6.1 The papers that still cross — written down, so the residual leak is actionable

The best balanced partition cannot get the cut to zero: the sharing graph is nearly fully
connected, so some papers appear in both halves no matter how the collections are dealt.

Recording *which* ones turns a caveat into a switch. Drop them from B and the held-out
read is strict; keep them and it isn't — either is defensible, but only if the list exists.

In [11]:
crossing = []
for identity, group in large_df[large_df["_paper_identity"].isin(shared_identities)].groupby(
        "_paper_identity"):
    sides = {assignment[uc] for uc in group["use_case_key"].unique()}
    if len(sides) > 1:
        for _, row in group.iterrows():
            crossing.append({
                "paper_identity": identity,
                "row_key": row["row_key"],
                "paper_id": row["paper_id"],
                "use_case_key": row["use_case_key"],
                "split_set": assignment[row["use_case_key"]],
                "label_positive": int(row["label_positive"]),
                "title": row["title"],
            })
crossing_frame = pd.DataFrame(crossing)

REPORT_DIR.mkdir(parents=True, exist_ok=True)
crossing_path = REPORT_DIR / "benchset_v1_ab_crossing_papers.csv"
crossing_frame.to_csv(crossing_path, index=False)

n_crossing_papers = crossing_frame["paper_identity"].nunique()
print(f"{n_crossing_papers:,} distinct papers sit on both sides of the boundary, "
      f"appearing as {len(crossing_frame):,} rows.")
print(f"That is {len(crossing_frame) / len(large_df):.2%} of the fittable corpus, "
      f"carrying {int(crossing_frame['label_positive'].sum())} positive labels.")
print(f"\nWritten to {crossing_path.name} - filter set_b on `row_key` for a strict "
      f"held-out evaluation.")
print("\nWhich collections they connect:")
print(crossing_frame.groupby(["split_set", "use_case_key"]).size()
      .rename("rows").to_string())

154 distinct papers sit on both sides of the boundary, appearing as 313 rows.
That is 0.22% of the fittable corpus, carrying 10 positive labels.

Written to benchset_v1_ab_crossing_papers.csv - filter set_b on `row_key` for a strict held-out evaluation.

Which collections they connect:
split_set    use_case_key           
large_set_a  synergy_brouwer_2019        99
             synergy_leenaars_2020       16
             synergy_moran_2021          21
             synergy_muthu_2021           4
             synergy_nelson_2002          2
             synergy_van_dis_2020        16
large_set_b  synergy_jeyaraman_2020       6
             synergy_radjenovic_2013      1
             synergy_walker_2018        140
             synergy_wassenaar_2017       8


## 7. Tier 1b lexical block

`build_lexical_features` (`scripts/lexical_features.py`), real briefs (default
`brief_map=None`). 22 columns, nothing fitted on labels, so it is safe to materialise
before any split. Prefixed `lex_` here since the function returns unprefixed names.

In [12]:
lexical = build_lexical_features(df).add_prefix("lex_")
assert lexical.shape[1] == 22, f"expected 22 lexical columns, got {lexical.shape[1]}"
print(f"Built {lexical.shape[1]} lex_* columns for {lexical.shape[0]:,} rows.")
df = pd.concat([df, lexical], axis=1)

null_rates = lexical.isna().mean().sort_values(ascending=False)
populated = null_rates[null_rates > 0]
if len(populated):
    print("\nNull rates (NaN is 'this brief specified no such terms', never 0):")
    print(populated.round(4).to_string())
else:
    print(
        "\nNo nulls in the block. In papers_fe.parquet this group averages a 4.5% null\n"
        "rate, because some TIRI pools specify no must/nice/exclude terms and the\n"
        "corresponding overlap columns are then NaN rather than 0. Every brief in this\n"
        "corpus specifies all three, so that NULL-not-zero path is never exercised here."
    )

Built 22 lex_* columns for 175,319 rows.

No nulls in the block. In papers_fe.parquet this group averages a 4.5% null
rate, because some TIRI pools specify no must/nice/exclude terms and the
corresponding overlap columns are then NaN rather than 0. Every brief in this
corpus specifies all three, so that NULL-not-zero path is never exercised here.


In [13]:
if BUILD_CONTROL:
    wrong_brief_map = derangements(sorted(df["use_case_key"].unique()), seed=0)
    control = build_lexical_features(df, brief_map=wrong_brief_map).add_prefix("lexctl_")
    df = pd.concat([df, control], axis=1)
    print(f"Built {control.shape[1]} lexctl_* columns (wrong-brief falsification control).")
else:
    print("BUILD_CONTROL is False - skipping the falsification-control columns.")

BUILD_CONTROL is False - skipping the falsification-control columns.


## 8. Cosine-to-brief, for two models and two brief variants

The zero-label ranker: cosine between each paper's vector and its own collection's brief
vector, plus the within-collection percentile rank (the convention `lexical_features.py`
already uses for `rank_bm25_*`). Both brief variants are computed: `cos_brief_*` uses the
full brief (including the review's abstract), `cos_briefpre_*` only what a customer has
before screening starts.

**The embeddings themselves are carried into the output, as `emb_jasper_0000...` /
`emb_qwen4b_0000...` scalar columns — the same layout `papers_fe.parquet` uses.**

This costs 2.71 GB across the three files (measured) to store vectors `embeddings_cache/`
already holds in 2.1 GB, and the temptation was to skip it and join at read time. That
would have been a mistake, for a reason worth writing down: every downstream consumer of
`papers_fe.parquet` — `06_baseline_logreg` through `10_logo_ensemble`, and ~10 scripts in
`scripts/` — selects its embedding block by **column-name prefix**, e.g. `06`'s
`EMBEDDING_COLS = [c for c in df.columns if c.startswith("emb_")]`.

Against a table with no such columns that expression returns `[]`, and nothing downstream
raises: `df[[]].to_numpy()` is a well-formed `(n, 0)` array, it `hstack`s cleanly with the
other blocks, and the run reports a plausible number fitted on metadata alone. Note the
asymmetry — `06` *does* check its non-embedding columns against an explicit expected list
and raises on a miss. The embedding side has no such tripwire, which is exactly why it
cannot be the side that goes missing.

A feature table that silently drops a feature block is worse than a large one. There is no
re-embedding cost either way (the cache is already built and `--verify` green), so the only
thing the slim version saved was disk — the cheapest thing on the table.

`embed_benchsets.load_paper_vectors` still does the join — keyed on
`(use_case_key, paper_id)`, since `paper_id` is unique only *within* a collection and a
global lookup would collide on the 2,092 papers that appear in more than one. The join
happens per output file at write time, so peak memory is one file's block rather than the
whole corpus.

**One difference from `papers_fe.parquet` that the prefix hides.** That table carries
*three* models (`jasper` + `qwen4b` + `qwen8b`, 8,704 dims); these carry **two** (4,608),
because only Jasper and Qwen3-4B were ever embedded for this corpus. `06_baseline_logreg`
selects with `startswith("emb_")` rather than per model, so its code runs unchanged against
either table — which is convenient, and also means the difference will not announce itself.
**A benchset ROC-AUC is therefore measured over 4,608 dimensions and a TIRI one over 8,704.**
Not a defect, but not a like-for-like comparison either.

In [14]:
for model_key in MODEL_KEYS:
    briefs = {v: load_brief_vectors(model_key, v) for v in BRIEF_VARIANT_PREFIX}
    columns = {f"{p}_{model_key}": np.full(len(df), np.nan) for p in BRIEF_VARIANT_PREFIX.values()}

    # Per collection, so peak memory is the largest collection's block rather than the
    # whole 175k x 2,560 corpus.
    for uc, rows in df.groupby("use_case_key", sort=False):
        vectors = load_paper_vectors(rows, model_key)
        vectors /= np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9
        where = df.index.get_indexer(rows.index)
        for variant, prefix in BRIEF_VARIANT_PREFIX.items():
            brief = briefs[variant][uc]
            columns[f"{prefix}_{model_key}"][where] = vectors @ (
                brief / (np.linalg.norm(brief) + 1e-9))

    for name, values in columns.items():
        df[name] = values
        df[f"rank_{name}"] = df.groupby("use_case_key")[name].rank(pct=True)
        print(f"{name}: mean={df[name].mean():.3f}  min={df[name].min():.3f}  "
              f"max={df[name].max():.3f}")

cos_cols = [c for c in df.columns if c.startswith(("cos_brief", "rank_cos_brief"))]
assert not df[cos_cols].isna().any().any(), "a cosine came out NaN - check for zero vectors"
print(f"\n{len(cos_cols)} brief-similarity columns, no NaNs.")

cos_brief_jasper: mean=0.551  min=0.199  max=0.964
cos_briefpre_jasper: mean=0.542  min=0.202  max=0.928


cos_brief_qwen4b: mean=0.393  min=0.060  max=0.918
cos_briefpre_qwen4b: mean=0.401  min=0.069  max=0.896

8 brief-similarity columns, no NaNs.


## 9. Raw admissible metadata

`year`, `paper_age`, `n_authors`, `citation_count` — pass-through or derived raw, NaN
preserved, never `fillna(0)`.

Admissible **only because every model here is fitted within one collection**. A pooled
model would have to drop them: `CONTEXT.md` records that a pooled model reads
`use_case_key` off metadata like this at high accuracy, and `03_eda_full_benchset_v1` §7
found publication year alone reaches 0.875 ROC-AUC on `synergy_moran_2021`. That is a
confound to be aware of even within a silo, not a feature to celebrate.

`has_abstract` is deliberately **not** carried: §2 made it constant `True`, and a constant
column is as useless as an all-NaN one. It is asserted constant, then dropped.

In [15]:
exported_year = pd.to_datetime(df["exported_at"]).dt.year
df["paper_age"] = (exported_year - df["year"]).clip(lower=0)


def count_authors(value):
    if pd.isna(value) or str(value).strip() == "":
        return np.nan
    return float(len(str(value).split(",")))


df["n_authors"] = df["authors"].map(count_authors)

assert bool(df["has_abstract"].all()), "has_abstract should be constant True after §2"
print("has_abstract is constant True after the §2 drop - asserted, then dropped as a feature.")

if exported_year.nunique() == 1:
    print(
        f"\nNOTE: every row shares one export timestamp ({exported_year.iloc[0]}), so "
        f"paper_age is an exact affine function of year and adds no information over it. "
        f"Carried for contract parity with 04_feature_engineering.ipynb; do not treat the "
        f"two as independent features."
    )

raw_metadata_cols = ["year", "paper_age", "n_authors", "citation_count"]
print("\nNull rates:")
print(df[raw_metadata_cols].isna().mean().rename("null_rate").round(4).to_string())

has_abstract is constant True after the §2 drop - asserted, then dropped as a feature.

NOTE: every row shares one export timestamp (2026), so paper_age is an exact affine function of year and adds no information over it. Carried for contract parity with 04_feature_engineering.ipynb; do not treat the two as independent features.

Null rates:
year              0.0868
paper_age         0.0868
n_authors         0.0986
citation_count    0.0959


## 10. Sanity checks, then write

`04_feature_engineering.ipynb` §10's assertions, plus the ones this corpus's structure
makes possible: that the three tiers partition the collections exactly, that the fittable
tier's paper overlap is no worse than §6.1 measured, and that the brief columns really are
constant within each collection (they are broadcast, and a cross-wired brief would silently
poison every lexical and cosine feature derived from it).

In [16]:
df["y"] = df["label_positive"].astype(bool)
df["split_set"] = df["use_case_key"].map(
    lambda uc: assignment.get(uc, "small_test")).astype("category")

assert len(df) == n_after_dedupe, f"row count drifted: {len(df)} != {n_after_dedupe}"
assert df["row_key"].is_unique, "row_key is not unique - the compound key is broken"

groups = {
    "identifiers/grouping": ["row_key", "paper_id", "use_case_key", "first_author"],
    "target": ["triage_label", "y"],
    "split": ["split_set"],
    "lexical (Tier 1b)": [c for c in df.columns if c.startswith("lex_")],
    "lexical control": [c for c in df.columns if c.startswith("lexctl_")],
    "brief similarity": cos_cols,
    "raw metadata": raw_metadata_cols,
}
final_cols = [c for cols in groups.values() for c in cols]
assert len(final_cols) == len(set(final_cols)), "a column is claimed by two groups"

all_nan = [c for c in final_cols if df[c].isna().all()]
assert not all_nan, f"all-NaN column(s): {all_nan}"

assert df["y"].nunique() == 2, "the target is constant - nothing can be fitted"

# A column with one value carries no information, whichever column it turns out to be.
# Same rule that removed `has_abstract` in §9, applied generally instead of by name -
# reported rather than silently dropped, because *which* column went constant is a fact
# about this corpus, not housekeeping.
feature_cols = [c for c in final_cols if c not in ("y", "triage_label")]
constant = [c for c in feature_cols
            if df[c].dtype.kind in "biufc" and df[c].nunique(dropna=True) <= 1]
if constant:
    print("Constant columns, dropped (no information to give a model):")
    for c in constant:
        print(f"  {c} = {df[c].dropna().iloc[0]!r} for all {len(df):,} rows")
    print(
        "\n  lex_has_exclude_terms is constant because all 28 briefs in this corpus\n"
        "  specify exclusion terms. On TIRI some pools specify none, which is why\n"
        "  04_feature_engineering.ipynb's exclusion check has a two-sided branch and\n"
        "  keeps the column. Here that branch is dead and the column is uninformative."
    )
    groups = {name: [c for c in cols if c not in constant] for name, cols in groups.items()}
    final_cols = [c for cols in groups.values() for c in cols]
    print()

numeric = df[final_cols].select_dtypes(include=[np.number])
n_inf = int(np.isinf(numeric.to_numpy(dtype=float)).sum())
assert n_inf == 0, f"found {n_inf} infinite values"

# The exclusion-overlap NaN pattern must match which briefs actually specified exclusions.
has_exclude = df.groupby("use_case_key")["terms_exclude"].first().map(len) > 0
for uc, expected in has_exclude.items():
    is_nan = df.loc[df["use_case_key"] == uc, "lex_overlap_excl_n"].isna().all()
    assert is_nan != bool(expected), (
        f"{uc}: exclusion terms present={bool(expected)} but lex_overlap_excl_n "
        f"all-NaN={is_nan} - these must disagree"
    )

# Briefs are broadcast per collection; a cross-wired one would poison every derived feature.
for col in ["objective", "problem_statement", "domain_industry", "domain_application"]:
    varying = df.groupby("use_case_key")[col].nunique(dropna=False)
    assert (varying <= 1).all(), (
        f"{col} varies within {varying[varying > 1].index.tolist()} - briefs are cross-wired"
    )

tiers = df.groupby("split_set", observed=True)["use_case_key"].nunique()
assert int(tiers.sum()) == df["use_case_key"].nunique() == 28, (
    f"tiers do not partition the 28 collections: {tiers.to_dict()}"
)
print("All sanity checks passed.")
print(f"\nCollections per output: {tiers.to_dict()}")

dropped_cols = [c for c in df.columns if c not in final_cols]
print(f"\nDropping {len(dropped_cols)} input-only columns not in the output contract "
      f"(still available in {DATA_PATH.name}):\n  {sorted(dropped_cols)}")

Constant columns, dropped (no information to give a model):
  lex_has_exclude_terms = np.True_ for all 175,319 rows

  lex_has_exclude_terms is constant because all 28 briefs in this corpus
  specify exclusion terms. On TIRI some pools specify none, which is why
  04_feature_engineering.ipynb's exclusion check has a two-sided branch and
  keeps the column. Here that branch is dead and the column is uninformative.



All sanity checks passed.

Collections per output: {'large_set_a': 8, 'large_set_b': 7, 'small_test': 13}

Dropping 19 input-only columns not in the output contract (still available in papers_benchset_v1.parquet):
  ['abstract', 'authors', 'brief_provenance', 'doi', 'domain_application', 'domain_industry', 'domain_technology_focus', 'exported_at', 'has_abstract', 'label_positive', 'lex_has_exclude_terms', 'objective', 'problem_statement', 'terms_exclude', 'terms_must_include', 'terms_nice_to_have', 'title', 'use_case_name', 'usecase_schema_version']


In [17]:
out = df[final_cols]
written = {}

# The embedding block is joined and exploded PER FILE, not once for the whole corpus:
# 175,319 x 4,608 float32 in a single frame is ~3.2 GB before pd.concat's copy, and there
# is no reason to hold set_a's vectors in memory while writing set_b's.
for split_set, block in out.groupby("split_set", observed=True):
    block = block.reset_index(drop=True)
    parts = [block]
    for model_key, expected_dim in EXPECTED_DIMS.items():
        vectors = load_paper_vectors(block, model_key)
        assert vectors.shape[1] == expected_dim, (
            f"{model_key}: expected {expected_dim} dims, cache gave {vectors.shape[1]}"
        )
        parts.append(pd.DataFrame(
            vectors, columns=[f"emb_{model_key}_{i:04d}" for i in range(expected_dim)]))
    wide = pd.concat(parts, axis=1)

    path = OUT_DIR / f"benchset_v1_{split_set}.parquet"
    wide.to_parquet(path, index=False)
    written[str(split_set)] = path
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"Wrote {path.name}: {len(wide):,} rows x {wide.shape[1]:,} cols, "
          f"{size_mb:,.1f} MB | {block['use_case_key'].nunique()} collections, "
          f"{int(block['y'].sum()):,} positives ({block['y'].mean():.2%})")
    del wide, parts

summary = pd.DataFrame(
    [(name, len(cols), round(float(out[cols].isna().mean().mean()), 4) if cols else 0.0)
     for name, cols in groups.items()]
    + [("embeddings", sum(EXPECTED_DIMS.values()), 0.0)],
    columns=["group", "n_cols", "mean_null_rate"],
)
print()
print(summary.to_string(index=False))
print(f"\nTotal on disk: "
      f"{sum(p.stat().st_size for p in written.values()) / 1024**3:.2f} GB — of which the "
      f"{sum(EXPECTED_DIMS.values()):,} embedding columns are ~98%. Carried rather than "
      f"joined at read time so that a fold pipeline copied from 06_baseline_logreg finds "
      f"the emb_* prefixes it slices on.")

Wrote benchset_v1_large_set_a.parquet: 62,229 rows x 4,648 cols, 981.5 MB | 8 collections, 1,362 positives (2.19%)


Wrote benchset_v1_large_set_b.parquet: 82,350 rows x 4,648 cols, 1,302.6 MB | 7 collections, 1,542 positives (1.87%)


Wrote benchset_v1_small_test.parquet: 30,740 rows x 4,648 cols, 486.9 MB | 13 collections, 357 positives (1.16%)

               group  n_cols  mean_null_rate
identifiers/grouping       4           0.000
              target       2           0.000
               split       1           0.000
   lexical (Tier 1b)      21           0.000
     lexical control       0           0.000
    brief similarity       8           0.000
        raw metadata       4           0.092
          embeddings    4608           0.000

Total on disk: 2.71 GB — of which the 4,608 embedding columns are ~98%. Carried rather than joined at read time so that a fold pipeline copied from 06_baseline_logreg finds the emb_* prefixes it slices on.


In [18]:
manifest = {
    "source": DATA_PATH.name,
    "row_reconciliation": {
        "start": n_start,
        "after_abstract_drop": n_after_abstract,
        "after_within_collection_dedupe": n_after_dedupe,
    },
    "positive_reconciliation": {
        "start": pos_start,
        "after_abstract_drop": pos_after_abstract,
        "after_within_collection_dedupe": pos_after_dedupe,
    },
    "rules": {
        "abstract": "rows with no abstract dropped, uniformly and with no exceptions",
        "min_positives": MIN_POSITIVES,
        "min_positives_per_test_fold": MIN_POS_PER_TEST_FOLD,
        "n_folds": N_FOLDS,
        "quarantined": QUARANTINE,
    },
    "partition_search": {
        "collections_searched": n_uc,
        "partitions_enumerated": 2 ** (n_uc - 1),
        "partitions_meeting_balance": len(candidates),
        "max_collection_count_diff": MAX_COUNT_DIFF,
        "max_positive_imbalance": MAX_POS_IMBALANCE,
        "max_row_imbalance": MAX_ROW_IMBALANCE,
        "objective": "minimise shared papers crossing A/B, tie-break on brief cosine",
        "shared_paper_links_total": float(total_links),
        "shared_paper_links_crossing": float(cut),
        "cross_boundary_brief_cosine": float(topical_cut),
        "positive_imbalance": float(pos_imbalance),
        "row_imbalance": float(row_imbalance),
    },
    "assignment": {uc: str(s) for uc, s in
                   df.groupby("use_case_key", observed=True)["split_set"].first().items()},
    "crossing_papers": {
        "distinct_papers": int(n_crossing_papers),
        "rows": int(len(crossing_frame)),
        "file": crossing_path.name,
    },
    "caveats": {
        "synergy_walker_2018": (
            f"{int(worst['pos_dropped'])} of its positives were dropped with their "
            f"abstract-less rows, which were {worst['enrichment']:.1f}x enriched for "
            f"positives. Scores on this collection are optimistic relative to the real "
            f"screening task."
        ),
        "embeddings": (
            "carried as emb_jasper_* / emb_qwen4b_* scalar columns, matching "
            "papers_fe.parquet, so prefix-slicing downstream code works unchanged. "
            "Rebuildable from data/processed/embeddings_cache via "
            "embed_benchsets.load_paper_vectors(df, model_key)"
        ),
        "paper_age": "affine in year (single export timestamp); not an independent feature",
    },
}
manifest_path = REPORT_DIR / "benchset_v1_split_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {manifest_path}")
print(json.dumps(manifest["partition_search"], indent=2))

Wrote ../../reports/benchset_v1_split_manifest.json
{
  "collections_searched": 15,
  "partitions_enumerated": 16384,
  "partitions_meeting_balance": 792,
  "max_collection_count_diff": 1,
  "max_positive_imbalance": 0.12,
  "max_row_imbalance": 0.15,
  "objective": "minimise shared papers crossing A/B, tie-break on brief cosine",
  "shared_paper_links_total": 2112.0,
  "shared_paper_links_crossing": 159.0,
  "cross_boundary_brief_cosine": 16.28481674194336,
  "positive_imbalance": 0.06198347107438017,
  "row_imbalance": 0.13916958894445253
}


## 11. Verify what was actually written

Reading the files back and re-running the fold check is not belt-and-braces: everything
above operated on an in-memory frame, and the claim being made is about three files on
disk. This checks the delivered artefact, not the intention.

In [19]:
# Scalar columns only for the reconciliation checks - pulling 4,608 embedding columns back
# into memory to count rows would read ~3 GB to answer a question the schema can answer.
scalar_cols = ["row_key", "paper_id", "use_case_key", "first_author", "y"]
reloaded = {name: pd.read_parquet(path, columns=scalar_cols)
            for name, path in written.items()}

for name, path in written.items():
    present = set(pq.ParquetFile(path).schema.names)
    for model_key, expected_dim in EXPECTED_DIMS.items():
        got = sum(1 for c in present if c.startswith(f"emb_{model_key}_"))
        assert got == expected_dim, f"{name}: emb_{model_key}_* is {got} wide, want {expected_dim}"
    missing = [c for c in final_cols if c not in present]
    assert not missing, f"{name}: missing contract columns {missing}"
print("On-disk schema: both embedding blocks full width in all three files, "
      "every contract column present.")
print("(checked from the parquet schema, so the assertion costs no memory)\n")

total_rows = sum(len(b) for b in reloaded.values())
total_pos = sum(int(b["y"].sum()) for b in reloaded.values())
assert total_rows == n_after_dedupe, f"{total_rows} != {n_after_dedupe}"
assert total_pos == pos_after_dedupe, f"{total_pos} != {pos_after_dedupe}"

seen = [set(b["use_case_key"].unique()) for b in reloaded.values()]
for x, y_ in itertools.combinations(seen, 2):
    assert not (x & y_), f"collection in two files: {x & y_}"
assert len(set().union(*seen)) == 28

keys_a = set(reloaded["large_set_a"]["row_key"])
keys_b = set(reloaded["large_set_b"]["row_key"])
assert not (keys_a & keys_b), "a row_key is in both A and B"

print(f"Reconciles: {n_start:,} rows -> {n_after_abstract:,} (abstract drop) -> "
      f"{n_after_dedupe:,} (dedupe) = {total_rows:,} written.")
print(f"            {pos_start:,} positives -> {pos_after_abstract:,} -> "
      f"{pos_after_dedupe:,} = {total_pos:,} written.")
print("28 collections partition cleanly across the three files; no row_key in two files.\n")

for name in ("large_set_a", "large_set_b"):
    block = reloaded[name]
    worst_fold = []
    for uc, sub in block.groupby("use_case_key"):
        y_uc = sub["y"].to_numpy().astype(int)
        splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
        worst_fold.append(min(int(y_uc[test].sum())
                              for _, test in splitter.split(sub, y_uc,
                                                            sub["first_author"].to_numpy())))
    assert min(worst_fold) >= MIN_POS_PER_TEST_FOLD, f"{name} not fittable as delivered"
    print(f"{name}: all {block['use_case_key'].nunique()} collections fit "
          f"{N_FOLDS}-fold as delivered (worst test fold has {min(worst_fold)} positives).")

small = reloaded["small_test"]
print(f"\nsmall_test: {small['use_case_key'].nunique()} collections, {len(small):,} rows, "
      f"{int(small['y'].sum()):,} positives — evaluation only, by construction. "
      f"Reporting a fitted CV score on these is the error this tier exists to prevent.")

On-disk schema: both embedding blocks full width in all three files, every contract column present.
(checked from the parquet schema, so the assertion costs no memory)

Reconciles: 181,199 rows -> 176,466 (abstract drop) -> 175,319 (dedupe) = 175,319 written.
            3,374 positives -> 3,270 -> 3,261 = 3,261 written.
28 collections partition cleanly across the three files; no row_key in two files.



large_set_a: all 8 collections fit 5-fold as delivered (worst test fold has 8 positives).


large_set_b: all 7 collections fit 5-fold as delivered (worst test fold has 9 positives).

small_test: 13 collections, 30,740 rows, 357 positives — evaluation only, by construction. Reporting a fitted CV score on these is the error this tier exists to prevent.


## 12. What was built, and what it is not

**Three files, one inclusion rule, both thresholds demonstrated rather than assumed.**

The honest limits, all of which travel in `benchset_v1_split_manifest.json`:

- **`synergy_walker_2018` is easier here than in reality.** Its dropped rows were 3.7x
  enriched for positives. Any score on it is optimistic — quote this alongside it.
- **A and B are not perfectly disjoint.** The crossing papers are enumerated in
  `benchset_v1_ab_crossing_papers.csv`; filter them out of B for a strict read.
- **A/B is dev vs held-out, not train vs test.** Fitting across collections is the
  experiment `CONTEXT.md` §1 rules out; both halves are for within-silo work.
- **`small_test` can never carry a cross-validated score.** Those 13 collections are for
  zero-shot ranking — cosine-to-brief, BM25, or a method carried in from elsewhere.
- **Metadata is admissible only within a silo**, and `year` is a live confound even there.
- **Embeddings are in these files** as `emb_jasper_*` / `emb_qwen4b_*`, matching
  `papers_fe.parquet`'s layout so that anything slicing on those prefixes works unchanged.
  That is ~98% of the ~3 GB on disk; `embed_benchsets.load_paper_vectors` rebuilds them
  from `embeddings_cache/` if a file is ever deleted.

Nothing here has been scored. The first number measured on `large_set_b` should be the
last one, and by then the method should already be settled on `large_set_a`.